In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\keras\engine\training_arrays_v1.py:37: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.23.2)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


In [2]:
# Load the original training data (assuming it's available)
train_data_df = pd.read_csv('X_cma.csv')

In [3]:
# Convert to numpy array
X_train = train_data_df.values

In [4]:
# Initialize and fit the scaler with the original training data
scaler = StandardScaler()
scaler.fit(X_train)

StandardScaler()

In [5]:
# Load the final saved model
model = load_model('final_model_CMA.h5')

In [6]:
# Load the new data from CSV
new_data_df = pd.read_csv('NewData CMA.csv')

In [7]:
# Check if there are any non-feature columns and drop them if necessary
if 'Unnamed: 0' in new_data_df.columns:
    new_data_df = new_data_df.drop(columns=['Unnamed: 0'])

In [8]:
# Convert to numpy array
new_data = new_data_df.values

In [9]:
# Scale the new data using the fitted scaler
new_data_scaled = scaler.transform(new_data)

In [10]:
# Define the sequence length used during training
seqlen = 10


In [11]:
# Generate sequences from the new data
def create_sequences(data, seq_length):
    sequences = []
    for i in range(len(data) - seq_length + 1):
        sequences.append(data[i:i+seq_length])
    return np.array(sequences)

In [12]:
new_data_sequences = create_sequences(new_data_scaled, seqlen)


In [13]:
# Make predictions
predictions = model.predict(new_data_sequences)

ValueError: Exception encountered when calling LSTMCell.call().

[1mDimensions must be equal, but are 174 and 176 for '{{node sequential_1_1/LSTM1_1/lstm_cell_1/MatMul}} = MatMul[T=DT_FLOAT, grad_a=false, grad_b=false, transpose_a=false, transpose_b=false](sequential_1_1/LSTM1_1/strided_slice_1, sequential_1_1/LSTM1_1/lstm_cell_1/Cast/ReadVariableOp)' with input shapes: [32,174], [176,80].[0m

Arguments received by LSTMCell.call():
  • inputs=tf.Tensor(shape=(32, 174), dtype=float32)
  • states=('tf.Tensor(shape=(32, 20), dtype=float32)', 'tf.Tensor(shape=(32, 20), dtype=float32)')
  • training=False

In [ ]:
# Convert probabilities to binary class predictions
threshold = 0.5
binary_predictions = (predictions > threshold).astype(int)


In [ ]:
# Combine the results into a DataFrame
results_df = pd.DataFrame({
    'Prediction_Probability': predictions.flatten(),
    'Predicted_Class': binary_predictions.flatten()
})


In [ ]:
# Save the results to a CSV file
results_df.to_csv('predictions.csv', index=False)

print("Predictions (Probabilities):", predictions)
print("Predictions (Binary):", binary_predictions)